In [ ]:
!pip install bertopic
!pip install datamapplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.0 MB/s eta 0:00:00
  Created wheel for Pyqtree: filename=Pyqtree-1.0.0-py3-none-any.whl size=5967 sha256=d08ef4092f83bcdc256d0dfc85b4ed5bea9f79778675172f235c2f95baf96756
  Stored in directory: /root/.cache/pip/wheels/63/e6/90/6e15bfb4299fd41f88a9affca879f44bde40d3dc6f398462a8
Successfully built Pyqtree


In [ ]:
# IMPORTS
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer, util
from umap import UMAP
import bertopic
import datamapplot
import pandas as pd
import glob
import os

In [ ]:
from google.colab import drive
drive.mount("/content/drive") #Datasets were uploaded to Google Drive and fetched from there

Mounted at /content/drive


In [ ]:
def create_dataframe(directory):
    causes = []
    effects = []

    #Glob to get all CSV files in the directory
    csv_files = glob.glob(os.path.join(directory, '*.csv'))

    for file in csv_files:
        try:
            df = pd.read_csv(file)

            #Check if "cause" and "effect" columns exist (lowercase)
            if 'cause' in df.columns and 'effect' in df.columns:
                # Replace missing values with 'NA' and convert to strings
                df['cause'] = df['cause'].fillna('NA').astype(str)
                df['effect'] = df['effect'].fillna('NA').astype(str)

                #Append the "cause" and "effect" data to the lists
                causes.extend(df['cause'].tolist())
                effects.extend(df['effect'].tolist())
            else:
                print(f"Skipping file {file} as it doesn't contain 'cause' and 'effect' columns")
        except Exception as e:
            print(f"Error reading file {file}: {e}")

    #Create a Dataframe from the causes and effects with columns "Cause" and "Effect"
    data = pd.DataFrame({'Cause': causes, 'Effect': effects})
    return data

data = create_dataframe('/content/drive/MyDrive/extracted_causes_effects/ThirdDataset_PDF_cause-effect') #Change dataset if needed

#Inspect dataset if needed:
# data.head(50)

# data["Cause"].str.len()

# data["Effect"].str.len().mean()

In [ ]:
causes = data['Cause'].to_list()
effects = data['Effect'].to_list()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words="english") #Define vectorizer model

In [ ]:
%%time
from sentence_transformers import SentenceTransformer

#Initialize embedding model
model_embedding = SentenceTransformer('all-MiniLM-L6-v2')

#Encode causes and effects
cause_embeddings = model_embedding.encode(causes)
effect_embeddings = model_embedding.encode(effects)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CPU times: user 54.9 s, sys: 755 ms, total: 55.6 s
Wall time: 1min 15s


In [ ]:
%%time
from umap import UMAP

#Initialize UMAP model with cosine as metric
umap_model = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine')

#Reduce dimensions for causes and effects
reduced_cause_embeddings = umap_model.fit_transform(cause_embeddings)
reduced_effect_embeddings = umap_model.fit_transform(effect_embeddings)


CPU times: user 51.2 s, sys: 413 ms, total: 51.6 s
Wall time: 44.7 s


In [ ]:
%%time
from bertopic import BERTopic

#Fit BERTopic model for causes
cause_model = BERTopic(
    n_gram_range=(1, 2),
    vectorizer_model=vectorizer_model,
    nr_topics='auto',
    min_topic_size=10,
    calculate_probabilities=True
).fit(causes, reduced_cause_embeddings)

#Fit BERTopic model for effects
effect_model = BERTopic(
    n_gram_range=(1, 2),
    vectorizer_model=vectorizer_model,
    nr_topics='auto',
    min_topic_size=10,
    calculate_probabilities=True
).fit(effects, reduced_effect_embeddings)


CPU times: user 52.8 s, sys: 458 ms, total: 53.2 s
Wall time: 41.6 s


In [ ]:
#Check the number of topics generated
data_topic_freq = cause_model.get_topic_freq()
topics_count = len(data_topic_freq) - 1
data_topic_freq

,Topic,Count
6,-1,2351
0,0,1287
33,1,1164
19,2,552
3,3,537
...,...,...
119,252,12
16,253,11
255,254,11
180,255,11


In [ ]:
cause_model.visualize_topics()

In [ ]:
effect_model.visualize_topics()

In [ ]:
cause_model.visualize_documents(causes, reduced_embeddings=reduced_cause_embeddings)

In [ ]:
effect_model.visualize_documents(effects, reduced_embeddings=reduced_effect_embeddings)

CPU times: user 818 ms, sys: 978 µs, total: 819 ms
Wall time: 863 ms


In [ ]:
#Create a DataMapPlot visualization if wanted
cause_model.visualize_document_datamap(causes, reduced_embeddings=reduced_cause_embeddings)

NameError: name 'cause_model' is not defined

In [ ]:
#Metrics

import numpy as np
#Extract cluster labels for metrics
cause_labels = np.array(cause_model.topics_)
effect_labels = np.array(effect_model.topics_)

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score, calinski_harabasz_score

#Evaluate clustering for causes
print("Evaluation Metrics for Causes Clustering:")
if len(set(cause_labels)) > 1: #Check if more than one cluster exists
    db_score = davies_bouldin_score(reduced_cause_embeddings, cause_labels)
    silhouette = silhouette_score(reduced_cause_embeddings, cause_labels)
    ch_score = calinski_harabasz_score(reduced_cause_embeddings, cause_labels)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")


Evaluation Metrics for Causes Clustering:
Davies-Bouldin Index: 31.248 (Lower is better)
Silhouette Score: -0.284 (Higher is better)
Calinski-Harabasz Score: 315.276 (Higher is better)


In [ ]:
#Evaluate clustering for effects
print("\nEvaluation Metrics for Effects Clustering:")
if len(set(effect_labels)) > 1:
    db_score = davies_bouldin_score(reduced_effect_embeddings, effect_labels)
    silhouette = silhouette_score(reduced_effect_embeddings, effect_labels)
    ch_score = calinski_harabasz_score(reduced_effect_embeddings, effect_labels)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")


Evaluation Metrics for Effects Clustering:
Davies-Bouldin Index: 12.736 (Lower is better)
Silhouette Score: -0.227 (Higher is better)
Calinski-Harabasz Score: 305.035 (Higher is better)
